# AdaBoost from scratch
Sections follow the lesson:
1. The 4-sample worked example (error, alpha, weight update, normalisation)
2. Error -> alpha mapping
3. Weighted decision stump (the weak learner)
4. The AdaBoost algorithm and the weighted vote
5. How the sample weights evolve (hard points gain attention)
6. Decision boundary: many axis-aligned stumps -> non-linear boundary
7. Exponential loss view: training bound `prod 2*sqrt(err*(1-err))`
8. Hyperparameters: `n_estimators` vs `learning_rate`, train vs test curves
9. Noise sensitivity, and `err > 0.5` (flip the learner)

In [ ]:
import os
import warnings

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
os.makedirs("plots", exist_ok=True)
rng = np.random.default_rng(42)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110})

## 1. Worked example (A, B = +1; C, D = -1; the first learner gets only C wrong)
Steps: init `w=1/N` -> weighted error -> `alpha = 0.5 ln((1-err)/err)` ->
`w <- w * exp(-alpha * y * h)` -> normalise.

In [ ]:
y4 = np.array([+1, +1, -1, -1])
h4 = np.array([+1, +1, +1, -1])            # sample C is misclassified
w4 = np.full(4, 1 / 4)
err4 = w4[h4 != y4].sum() / w4.sum()
alpha4 = 0.5 * np.log((1 - err4) / err4)
w_unnorm = w4 * np.exp(-alpha4 * y4 * h4)
w_new = w_unnorm / w_unnorm.sum()
print("err = %.3f | alpha = %.3f | e^-a = %.3f | e^a = %.3f" % (err4, alpha4, np.exp(-alpha4), np.exp(alpha4)))
print("unnormalised:", np.round(w_unnorm, 3), "sum = %.3f" % w_unnorm.sum())
print("normalised  :", np.round(w_new, 3), "(C carries half of the attention)")
assert abs(w_new[2] - 0.5) < 1e-12

# vote example from the lesson: alphas .8,.6,.2 with votes +1,+1,-1
print("weighted vote:", 0.8 * 1 + 0.6 * 1 + 0.2 * -1, "-> sign =", int(np.sign(1.2)))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
names = list("ABCD")
cols = ["tab:green" if h4[i] == y4[i] else "tab:red" for i in range(4)]
ax[0].bar(names, w4, color=cols); ax[0].set(title="Round 1 weights", ylabel="weight")
ax[1].bar(names, w_new, color=cols); ax[1].set(title="Round 2 weights (red = was wrong)")
fig.tight_layout(); fig.savefig("plots/01_worked_example.png"); plt.close(fig)

## 2. Error -> alpha
`err=0.5` gives `alpha=0` (a coin flip has no voice), `err->0` gives a huge vote,
and `err>0.5` gives a *negative* alpha (the learner is anti-informative).

In [ ]:
for e in (0.01, 0.10, 0.25, 0.50):
    print("err %.2f -> alpha %.2f" % (e, 0.5 * np.log((1 - e) / e)))
e_grid = np.linspace(0.01, 0.99, 200)
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(e_grid, 0.5 * np.log((1 - e_grid) / e_grid)); ax.axhline(0, c="k", lw=0.8); ax.axvline(0.5, c="gray", ls="--")
ax.set(xlabel="weighted error", ylabel="alpha", title="Vote weight vs error")
fig.tight_layout(); fig.savefig("plots/02_alpha_vs_error.png"); plt.close(fig)

## 3. Weighted decision stump
For every feature, sort the points and sweep the threshold; the weighted error of a split is
computed from cumulative sums in O(N log N) per feature (so O(N*D) scan after sorting).
Prediction is `polarity * sign(x_j - thr)` with polarity in {+1,-1}.

In [ ]:
class Stump:
    def fit(self, X, y, w):
        n, d = X.shape
        w = w / w.sum()
        best = (np.inf, 0, 0.0, 1)
        for j in range(d):
            o = np.argsort(X[:, j], kind="stable")
            xs, ys, ws = X[o, j], y[o], w[o]
            # predict +1 for x > thr (polarity +1): errors = positives to the left + negatives to the right
            pos_left = np.cumsum(ws * (ys == 1))
            neg_left = np.cumsum(ws * (ys == -1))
            err_pol1 = pos_left + (neg_left[-1] - neg_left)           # for thr after index i
            valid = xs[:-1] < xs[1:]
            for pol, errs in ((1, err_pol1), (-1, 1 - err_pol1)):
                e = np.where(valid, errs[:-1], np.inf)
                i = int(np.argmin(e))
                if e[i] < best[0]:
                    best = (e[i], j, 0.5 * (xs[i] + xs[i + 1]), pol)
        self.err, self.j, self.thr, self.pol = best
        return self

    def predict(self, X):
        return self.pol * np.where(X[:, self.j] > self.thr, 1, -1)


# 2-D toy data: ring-shaped classes (not separable by any single line)
def make_ring(n, noise=0.0, rng=rng):
    X = rng.normal(size=(n, 2))
    y = np.where(X[:, 0] ** 2 + X[:, 1] ** 2 > 1.4, 1, -1)
    X = X + 0.15 * rng.normal(size=X.shape)
    flip = rng.random(n) < noise
    return X, np.where(flip, -y, y)

Xtr, ytr = make_ring(400); Xte, yte = make_ring(2000)
s = Stump().fit(Xtr, ytr, np.ones(len(ytr)))
print("single stump: feature %d thr %.2f pol %+d | train acc %.3f test acc %.3f"
      % (s.j, s.thr, s.pol, (s.predict(Xtr) == ytr).mean(), (s.predict(Xte) == yte).mean()))

## 4. AdaBoost (discrete / SAMME for two classes)
Each round: fit stump on weighted data, `err`, `alpha = lr * 0.5 ln((1-err)/err)`, reweight, normalise.
`decision_function` is the margin `F(x)=sum alpha_t h_t(x)`; prediction is its sign.
A learner with `err >= 0.5` is flipped (its predictions negated, err -> 1-err) as the lesson suggests;
a perfect stump (`err=0`) is clipped with a small epsilon.

In [ ]:
class AdaBoost:
    def __init__(self, n_estimators=50, learning_rate=1.0):
        self.T, self.lr = n_estimators, learning_rate

    def fit(self, X, y):
        n = len(y)
        w = np.full(n, 1 / n)
        self.stumps, self.alphas, self.errs = [], [], []
        self.w_hist = [w.copy()]
        for _ in range(self.T):
            st = Stump().fit(X, y, w)
            pred = st.predict(X)
            err = np.clip(w[pred != y].sum(), 1e-10, 1 - 1e-10)
            if err > 0.5:                        # flip the learner instead of a negative vote
                st.pol *= -1; pred = -pred; err = 1 - err
            alpha = self.lr * 0.5 * np.log((1 - err) / err)
            w = w * np.exp(-alpha * y * pred)
            w /= w.sum()
            self.stumps.append(st); self.alphas.append(alpha); self.errs.append(err); self.w_hist.append(w.copy())
        self.alphas = np.array(self.alphas); self.errs = np.array(self.errs)
        return self

    def staged_decision(self, X):
        F = np.zeros(len(X))
        for a, st in zip(self.alphas, self.stumps):
            F = F + a * st.predict(X)
            yield F.copy()

    def decision_function(self, X, m=None):
        F = np.zeros(len(X))
        for a, st in list(zip(self.alphas, self.stumps))[:m]:
            F += a * st.predict(X)
        return F

    def predict(self, X, m=None):
        return np.where(self.decision_function(X, m) >= 0, 1, -1)


ada = AdaBoost(60).fit(Xtr, ytr)
print("AdaBoost(60 stumps): train acc %.3f test acc %.3f" % ((ada.predict(Xtr) == ytr).mean(), (ada.predict(Xte) == yte).mean()))
print("first 8 errors:", np.round(ada.errs[:8], 3))
print("first 8 alphas:", np.round(ada.alphas[:8], 3))
assert (ada.errs < 0.5).all()

## 5. Sample weights adapt
Marker area is proportional to the current weight. After a few rounds the points near the
boundary (hard ones) carry most of the weight, and the stump error on the *reweighted* data
stays close to (but under) 0.5 - each new stump is forced to attack what the others miss.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharex=True, sharey=True)
for ax, t in zip(axes, (0, 1, 5, 30)):
    w = ada.w_hist[t]
    ax.scatter(Xtr[:, 0], Xtr[:, 1], s=2 + 4000 * w, c=ytr, cmap="coolwarm", alpha=0.7, edgecolor="k", linewidth=0.2)
    ax.set(title="round %d (max w = %.3f)" % (t, w.max()))
fig.tight_layout(); fig.savefig("plots/03_weight_evolution.png"); plt.close(fig)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(ada.errs, "o-", ms=3); ax[0].axhline(0.5, c="gray", ls="--"); ax[0].set(xlabel="round", ylabel="weighted error of stump", title="Stump error on reweighted data")
ax[1].plot(ada.alphas, "o-", ms=3, c="tab:orange"); ax[1].set(xlabel="round", ylabel="alpha", title="Vote weights")
fig.tight_layout(); fig.savefig("plots/04_errors_alphas.png"); plt.close(fig)
print("effective sample size N_eff=1/sum(w^2): round0 %.0f -> round30 %.0f"
      % (1 / (ada.w_hist[0] ** 2).sum(), 1 / (ada.w_hist[30] ** 2).sum()))

## 6. Decision boundary: weighted sum of stumps
One stump draws a single axis-aligned line; the weighted combination builds a blocky,
non-linear region that can wrap around the ring.

In [ ]:
gx, gy = np.meshgrid(np.linspace(-3.2, 3.2, 200), np.linspace(-3.2, 3.2, 200))
G = np.c_[gx.ravel(), gy.ravel()]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.7), sharex=True, sharey=True)
for ax, m in zip(axes, (1, 3, 10, 60)):
    Z = ada.decision_function(G, m).reshape(gx.shape)
    ax.contourf(gx, gy, Z, levels=np.linspace(-Z.__abs__().max(), Z.__abs__().max(), 21), cmap="coolwarm", alpha=0.6)
    ax.contour(gx, gy, Z, levels=[0], colors="k", linewidths=1.2)
    ax.scatter(Xtr[:, 0], Xtr[:, 1], c=ytr, cmap="coolwarm", s=6, edgecolor="k", linewidth=0.2)
    ax.set(title="%d stump(s): test acc %.3f" % (m, (ada.predict(Xte, m) == yte).mean()))
fig.tight_layout(); fig.savefig("plots/05_decision_boundary.png"); plt.close(fig)

## 7. Exponential loss view (Friedman-Hastie-Tibshirani)
AdaBoost is forward stagewise minimisation of `L = exp(-y F(x))`. Consequences we can verify:
* the average exponential loss equals the product of the normalisers `Z_t = 2 sqrt(err_t (1-err_t))`
  (when `learning_rate = 1`), so it falls geometrically whenever `err_t < 0.5`;
* the training 0/1 error is bounded by that same product;
* the update `w <- w exp(-alpha y h)` is the gradient of exp loss at the current margin.

In [ ]:
ada1 = AdaBoost(60, 1.0).fit(Xtr, ytr)
exp_loss, train_err = [], []
for F in ada1.staged_decision(Xtr):
    exp_loss.append(np.exp(-ytr * F).mean()); train_err.append((np.where(F >= 0, 1, -1) != ytr).mean())
bound = np.cumprod(2 * np.sqrt(ada1.errs * (1 - ada1.errs)))
print("max |exp-loss - prod Z_t| = %.2e" % np.abs(np.array(exp_loss) - bound).max())
print("train error <= bound at every round:", bool(np.all(np.array(train_err) <= bound + 1e-12)))
print("exp-loss decreasing every round:", bool(np.all(np.diff(exp_loss) < 0)))
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.semilogy(exp_loss, label="mean exp loss"); ax.semilogy(bound, "--", label=r"$\prod 2\sqrt{err(1-err)}$")
ax.semilogy(np.maximum(train_err, 1e-4), label="train 0/1 error"); ax.set(xlabel="rounds", title="Exponential-loss bound"); ax.legend()
fig.tight_layout(); fig.savefig("plots/06_exp_loss_bound.png"); plt.close(fig)

t = np.linspace(-2, 2, 200)
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(t, np.exp(-t), label="exponential"); ax.plot(t, np.log(1 + np.exp(-t)), label="log-loss"); ax.plot(t, (t < 0).astype(float), label="0/1")
ax.set(xlabel="margin y*F(x)", ylim=(0, 4), title="Exp loss explodes for confident mistakes"); ax.legend()
fig.tight_layout(); fig.savefig("plots/07_loss_shapes.png"); plt.close(fig)

## 8. Hyperparameters: `n_estimators` and `learning_rate`
Smaller learning rate = slower convergence, and needs more estimators for the same fit.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for lr in (1.0, 0.3, 0.1):
    m = AdaBoost(200, lr).fit(Xtr, ytr)
    tr = [(np.where(F >= 0, 1, -1) == ytr).mean() for F in m.staged_decision(Xtr)]
    te = [(np.where(F >= 0, 1, -1) == yte).mean() for F in m.staged_decision(Xte)]
    ax[0].plot(tr, label="lr=%.1f" % lr); ax[1].plot(te, label="lr=%.1f" % lr)
    print("lr=%.1f: 200 stumps train %.3f test %.3f | best test %.3f at round %d" % (lr, tr[-1], te[-1], max(te), int(np.argmax(te)) + 1))
ax[0].set(title="Train accuracy", xlabel="n_estimators"); ax[1].set(title="Test accuracy", xlabel="n_estimators"); ax[0].legend()
fig.tight_layout(); fig.savefig("plots/08_lr_vs_estimators.png"); plt.close(fig)

## 9. Noise sensitivity (mislabelled points keep gaining weight)
With 15% flipped labels the train accuracy keeps climbing while clean test accuracy stalls
or degrades, and the largest weights sit on the flipped points.

In [ ]:
rng2 = np.random.default_rng(7)
Xc, yc = make_ring(400, rng=rng2)                                # clean data, then flip 15% explicitly
flipped = rng2.random(400) < 0.15
yn = np.where(flipped, -yc, yc)
noisy = AdaBoost(300, 1.0).fit(Xc, yn)
tr = [(np.where(F >= 0, 1, -1) == yn).mean() for F in noisy.staged_decision(Xc)]
te = [(np.where(F >= 0, 1, -1) == yte).mean() for F in noisy.staged_decision(Xte)]
wf = noisy.w_hist[-1]
top = np.argsort(wf)[-20:]
print("noisy labels: train acc (noisy) %.3f -> %.3f | clean test acc best %.3f at round %d, final %.3f"
      % (tr[0], tr[-1], max(te), int(np.argmax(te)) + 1, te[-1]))
print("share of top-20 weights on flipped points: %.2f (base rate %.2f)" % (flipped[top].mean(), flipped.mean()))
print("weight mass on flipped points after 300 rounds: %.2f" % wf[flipped].sum())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(tr, label="train (noisy labels)"); ax[0].plot(te, label="test (clean)"); ax[0].set(xlabel="rounds", title="AdaBoost with 15% label noise"); ax[0].legend()
ax[1].scatter(Xc[:, 0], Xc[:, 1], s=4 + 3000 * wf, c=np.where(flipped, "tab:red", "tab:blue"), alpha=0.6)
ax[1].set(title="Final weights (red = flipped label)")
fig.tight_layout(); fig.savefig("plots/09_noise_sensitivity.png"); plt.close(fig)

### Worse-than-chance learner
If `err > 0.5` the formula gives a negative alpha; flipping the learner is equivalent.

In [ ]:
err_bad = 0.8
a_neg = 0.5 * np.log((1 - err_bad) / err_bad)
a_flip = 0.5 * np.log(err_bad / (1 - err_bad))
print("err=0.8 -> alpha %.3f (negative vote) == flipped learner with err=0.2, alpha %.3f" % (a_neg, a_flip))

## Summary
* AdaBoost = weak learners (stumps) + adaptive sample reweighting + alpha-weighted vote.
* `alpha = 0.5 ln((1-err)/err)`; wrong points are multiplied by `e^alpha`, right ones by `e^-alpha`.
* It is forward stagewise minimisation of exponential loss: mean loss = `prod Z_t` and bounds training error.
* Reduces bias; many stumps form a non-linear boundary. Small learning rate needs more rounds.
* Exponential loss punishes confident mistakes, so noisy labels / outliers dominate the weights.